# Човек в контрола: Предварителни стени за действия, категоризация на риска и журнал на одита

README за този урок въвежда Човек в контрола с кратък фрагмент, който пита потребителя да `APPROVE` или `REJECT` след като агентът вече е дал отговор. Този модел е добър начален пункт, но реалните реализации на HITL на практика често изискват още три неща:

1. **предварителна стена за действие**, която се изпълнява **преди** агентът да извърши рискова стъпка, за да се поддържат под контрол разходите, необратимостта и забавянето.
2. **категоризация на риска**, така че действия с нисък риск да се изпълняват автоматично, действия със среден риск да се одобряват на групи, а само действия с висок риск да изискват човешка намеса.
3. **журнал на одита плюс цикъл за преразглеждане**, така че всяко решение на стената да се записва като JSONL, а отхвърлянето да предизвиква повторно подканяне на агента с структурирана причина, вместо просто да се извежда `Revising...`.

Този запис в тетрадката изгражда всяка от тези функции върху същите примитиви като `06-system-message-framework.ipynb`. Изпълнява се от край до край в `DEMO_MODE = True` (не се изисква интерактивно въвеждане) или с реални `input()` подкани, когато `DEMO_MODE = False`. Забележка: в DEMO_MODE повторният опит по третата цел е скриптиран, така че механиката на цикъла да е видима изцяло. Реалното преразглеждане, базирано на ревизия, изисква `DEMO_MODE = False` и оператор.

**Извън обхвата (обработва се в други уроци):** удостоверяване и контрол на достъпа (Урок 06 README, заплаха 2), междинен слой за повикване на инструменти (Урок 14 MAF в дълбочина), модели за спор между няколко агента.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Модел 1: Предварителен контрол

README фрагментът HITL първо извиква агента, след което иска от потребителя да одобри резултата. Това е **пост-действие** поток. Агентът вече е изпълнил, така че разходът за извикване на LLM вече е платен и всяка странична реакция (изпратен имейл, записана база данни, публикуван коментар) вече се е случила.

**Предварителен** поток вмъква контрол преди агентът да изпълни рисковата стъпка. Агентът предлага действието, контролът решава дали да се изпълни и само след одобрение се случва страничният ефект.

| Аспект | Одобрение след действието (README фрагмент) | Предварителен контрол (този бележник) |
|---|---|---|
| Кога се изпълнява одобрението? | След като агентът е произвел изход | Преди да се изпълни какъвто и да е страничен ефект |
| Разходи за LLM при отказ | Вече платени | Платени само за предложението, не за действието |
| Необратими странични ефекти | Възможни (действието вече се е случило) | Предотвратени |
| Яснота на одита | Одобрението е отпечатана команда | Одобрението е JSON запис с времеви печат, действие, причина |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Модел 2: Рисково разслояване

Не всяко действие изисква човешко одобрение. Само за четене заявка към публично API има различни залози от изпращане на имейл на клиент. Отнасянето към двете еднакво губи внимание на оператора и забавя агента.

Прост модел с 3 нива:

| Ниво | Примери | Поток на одобрение |
|---|---|---|
| `ниско` (само за четене) | Търсене в база знания, преглед на опции за полети, взимане на публична уеб страница | Автоматично изпълнение, записано за одит |
| `средно` (евтина мутация) | Кеширане на резултат, увеличаване на брояч, планиране на напомняне | Автоматично изпълнение, но с дневен преглед на партиди |
| `високо` (външно насочено или необратимо) | Изпращане на имейл, таксуване на карта, публикуване в публичен канал | Блокира се до човешко одобрение |

Това е едно разслояване. Производствените системи често използват по-гранулирани нива (например нива на разрешения в AWS IAM, нива за достъп основани на роли). Версията с 3 нива по-долу е най-малката полезна версия за агент, който смесва действия само за четене и с странични ефекти.

Класификаторът по-долу използва ключови думи за евристика, за да остане демонстрацията детерминирана и евтина. В производствена система бихте заменили това с научен класификатор или двигател за политики.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Модел 3: Регистър на одит и цикъл на ревизия

`print("Response approved.")` не е регистър на одит. За доверие, всяко решение на портата трябва да се записва като структурирано събитие, което по-късно можете да заявявате, възпроизвеждате или да прикачвате към преглед на инциденти.

Два компонента:

1. **Само добавяне JSONL.** Един ред на решение, със стойност за време, действие, ниво, решение, причина. Лесно се търси с grep, лесно се изпраща към реално хранилище за логове по-късно.
2. **Цикъл на ревизия при отказване.** Когато вратата връща `deny`, агентът отново се подканя със съобщението за отказ в контекста, така че следващото предложение да може да избегне проблема.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Допълнителни ресурси

Няколко други обществени проекта реализират вариации на тези HITL модели. Сравнете подходите, за да намерите какво пасва на вашия стек:

- **LangChain** програмни обвивки за инструменти с човешки контрол ([документация](https://python.langchain.com/docs/integrations/tools/human_tools)): програмни обвивки за инструменти, които спират изпълнението за човешка намеса.
- **AutoGen** `UserProxyAgent` ([документация v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ преработи това): използва роля на агент, специално за представяне на човека в разговори с много агенти.
- **Semantic Kernel** филтри на функции ([документация](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): междинен софтуер, който работи около всеки повикване на функция, подходящ за контролна логика.

Всеки проект борави с трите подпатерна по различен начин: LangChain ги обвива като инструменти, AutoGen използва роля на агент, Semantic Kernel използва междинни филтри. Прочетете една или две реализации изцяло, преди да изберете дизайн за собствения си агент.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
